# Notebook 02: robustness and percolation

How far does the rail network have to degrade before it falls apart, and which single stations do the most damage? This notebook works through RQ1 with the frozen Phase 2 outputs. It reads committed CSV files, runs in under a minute, and does not rerun any experiment.

Inputs (frozen 8 October 2026):

- `results/percolation/percolation_*.csv`: eight removal curves with 92 fractions and 100 seeded runs each
- `results/percolation/percolation_critical_fractions.csv` and `percolation_conclusions.csv`: both collapse estimates and the comparisons
- `results/percolation/vulnerability_*.csv`: all 86 single-station triggers over 81 tolerances
- `results/uncertainty/random_failure_ci.csv`: the interval for the random node curve
- `figures/fig_percolation_curves.png` and the other committed family figures

Regenerate the inputs with `make percolation PYTHON=.venv/bin/python`.

## Setup

Data paths come from `transperth.config` and the plot style from `transperth.plotting`, so the notebook reads the same files and draws in the same style as the runners. The small path bootstrap lets the notebook run from the repository root and from `notebooks/`, which is the working directory `nbconvert` uses.

In [ ]:
import sys
from pathlib import Path

project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(project_root / "src") not in sys.path:
    sys.path.insert(0, str(project_root / "src"))

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

from transperth.config import FIGURES_DIR, RESULTS_DIR
from transperth.failure import critical_fraction, gcc_threshold_crossing
from transperth.plotting import STYLE

%matplotlib inline
plt.rcParams.update(STYLE)

PERCOLATION_DIR = RESULTS_DIR / "percolation"
UNCERTAINTY_DIR = RESULTS_DIR / "uncertainty"
NOTEBOOK_FIGURES = FIGURES_DIR / "notebook02"
NOTEBOOK_FIGURES.mkdir(parents=True, exist_ok=True)


def save_copy(fig, name):
    """Save a PNG copy under figures/notebook02 and return its path."""
    path = NOTEBOOK_FIGURES / name
    fig.savefig(path, dpi=150, bbox_inches="tight")
    return path

## What the curves measure

A removal fraction $f$ removes that fraction of the 86 stations, or of the 85 edges, in one batch. Each of the 92 grid values runs with 100 seeds, and every run records the largest connected component as a fraction of the original 86 stations:

$$g(f) = \frac{\text{largest component after the removal}}{86}$$

The second-largest component feeds the susceptibility

$$\chi(f) = \frac{\overline{\mathrm{lcc}^2(f)}}{n_0}$$

where $\mathrm{lcc}$ is the second-largest component size and $n_0 = 86$. Both $g(f)$ and $\chi(f)$ are averaged over the 100 seeds, since a single run is noisy.

The two collapse estimates in `percolation_critical_fractions.csv` read the same curves in different ways:

- `critical_fraction` is the smallest $f$ at which $\chi(f)$ peaks.
- `gcc_threshold_fraction` is the first $f$ where the seed-mean $g(f)$ reaches 0.5, interpolated linearly between the two surrounding grid points.

The estimates disagree on this network. The susceptibility peak fires while the largest component still holds 35 to 67 percent of the stations, because the second-largest component is widest before the split reaches the largest one. The 0.5 crossing matches the question RQ1 asks (where does the largest component halve), so the notebook keeps both visible. The full definitions are in `docs/model.md` and `docs/metrics.md`.

In [ ]:
def curve_slug(component, attack, measure):
    """Slug of the frozen curve for a component, attack and target measure."""
    if attack == "random":
        prefix = "random_edge" if component == "edge" else "random"
        return f"{prefix}_degree"
    prefix = "targeted_edge" if component == "edge" else "targeted"
    return f"{prefix}_{measure}"


def measure_of(slug):
    """Measure label of a curve slug, with random attacks sharing one label."""
    return "random" if slug.startswith("random") else slug.split("_")[-1]


SLUGS = [
    curve_slug(component, attack, measure)
    for component in ("node", "edge")
    for attack, measure in [
        ("random", "degree"),
        ("targeted", "degree"),
        ("targeted", "betweenness"),
        ("targeted", "flow"),
    ]
]
curves = {
    slug: pd.read_csv(PERCOLATION_DIR / f"percolation_{slug}.csv") for slug in SLUGS
}
estimates = pd.read_csv(
    PERCOLATION_DIR / "percolation_critical_fractions.csv"
).set_index("slug")
interval = pd.read_csv(UNCERTAINTY_DIR / "random_failure_ci.csv").set_index("fraction")

print(f"{len(curves)} curves, {sum(len(curve) for curve in curves.values()):,} seeded runs")

## Collapse curves

Random removal drops stations without regard to their position. Targeted removal ranks stations or edges by degree, betweenness load or demand flow and removes the top of the list first. The two panels below show the seed-mean curve of each attack on the two layers.

In [ ]:
PANELS = {
    "node removal": [
        "random_degree",
        "targeted_degree",
        "targeted_betweenness",
        "targeted_flow",
    ],
    "edge removal": [
        "random_edge_degree",
        "targeted_edge_degree",
        "targeted_edge_betweenness",
        "targeted_edge_flow",
    ],
}
COLORS = {
    "random": "#7f7f7f",
    "degree": "#1f77b4",
    "betweenness": "#d62728",
    "flow": "#2ca02c",
}
LABELS = {
    "random": "random",
    "degree": "targeted degree",
    "betweenness": "targeted betweenness",
    "flow": "targeted flow",
}

fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.2), sharey=True)
for ax, (title, slugs) in zip(axes, PANELS.items()):
    for slug in slugs:
        measure = measure_of(slug)
        mean = curves[slug].groupby("fraction")["gcc_fraction"].mean()
        crossing = float(estimates.loc[slug, "gcc_threshold_fraction"])
        ax.plot(mean.index, mean, color=COLORS[measure], lw=1.5, label=LABELS[measure])
        ax.plot([crossing], [0.5], "o", ms=5.5, color=COLORS[measure], mec="white", mew=0.7, zorder=5)
    ax.axhline(0.5, color="0.55", lw=0.9, ls=":")
    ax.set_title(title)
    ax.set_xlabel(r"removed fraction $f$")
    ax.set_xlim(0, 0.5)
    ax.set_ylim(-0.03, 1.05)
axes[0].set_ylabel(r"largest component fraction $g(f)$")
axes[0].fill_between(
    interval.index, interval["low"], interval["high"], color=COLORS["random"], alpha=0.22,
    label="random node, 95% interval",
)
axes[0].legend(fontsize=8, loc="upper right")
save_copy(fig, "percolation_overview.png")
plt.show()

Lines are seed means over the 100 runs per fraction. Dots mark the `gcc_threshold_fraction` crossing of each curve, where the largest component falls to half; the shaded band is the interval for the random node curve from the uncertainty runs (`results/uncertainty/random_failure_ci.csv`, 1,000 seeded runs per fraction). Axes stop at $f = 0.5$, where the transition is long over. Saved copy: `figures/notebook02/percolation_overview.png`.

In [ ]:
rows = []
for slug, curve in curves.items():
    frozen = estimates.loc[slug]
    rows.append(
        {
            "curve": slug,
            "chi peak (frozen)": float(frozen["critical_fraction"]),
            "chi peak (recomputed)": critical_fraction(curve),
            "gcc 0.5 (frozen)": float(frozen["gcc_threshold_fraction"]),
            "gcc 0.5 (recomputed)": gcc_threshold_crossing(curve),
        }
    )
checks = pd.DataFrame(rows)
checks["agree"] = np.isclose(
    checks["chi peak (frozen)"], checks["chi peak (recomputed)"]
) & np.isclose(checks["gcc 0.5 (frozen)"], checks["gcc 0.5 (recomputed)"])
checks.round(4)

The recomputed columns call `transperth.failure.critical_fraction` and `gcc_threshold_crossing` on the raw curves. The frozen estimates come out of the same definitions, which checks that the committed numbers and the packaged engine agree.

In [ ]:
conclusions = pd.read_csv(PERCOLATION_DIR / "percolation_conclusions.csv")
show = conclusions[
    [
        "question",
        "first_curve",
        "second_curve",
        "first_gcc_threshold_fraction",
        "second_gcc_threshold_fraction",
        "gcc_threshold_ratio",
    ]
].rename(
    columns={
        "first_gcc_threshold_fraction": "first gcc 0.5",
        "second_gcc_threshold_fraction": "second gcc 0.5",
        "gcc_threshold_ratio": "ratio",
    }
)
show.round(3)

## Interactive collapse viewer

Choose a component and an attack, then drag the removal fraction. Random attacks ignore the measure dropdown: without a target list every station or edge is equally likely, so each layer has one random curve and the runner stores it under `degree`. For targeted attacks the measure decides which nodes or edges go first.

The vertical lines mark the two frozen estimates of the selected curve. The text box reports the seed mean, the seed envelope and the isolated count at the selected fraction. On the random node curve, fractions that fall on the coarser uncertainty grid also carry its interval.

In [ ]:
component_toggle = widgets.ToggleButtons(
    options=[("Node", "node"), ("Edge", "edge")], description="Component"
)
attack_toggle = widgets.ToggleButtons(
    options=[("Random", "random"), ("Targeted", "targeted")],
    value="targeted",
    description="Attack",
)
measure_dropdown = widgets.Dropdown(
    options=["degree", "betweenness", "flow"], value="degree", description="Measure"
)
fraction_slider = widgets.FloatSlider(
    value=0.10,
    min=0.0,
    max=1.0,
    step=0.01,
    description=r"Removed $f$",
    readout_format=".2f",
    continuous_update=False,
)


def value_at(series, fraction):
    """Value of a fraction-indexed series at a slider position."""
    match = np.isclose(series.index.to_numpy(dtype=float), fraction, atol=1e-9)
    return float(series[match].iloc[0])


def collapse_view(component, attack, measure, fraction):
    slug = curve_slug(component, attack, measure)
    curve = curves[slug]
    grouped = curve.groupby("fraction")["gcc_fraction"]
    mean = grouped.mean()
    low = grouped.quantile(0.05)
    high = grouped.quantile(0.95)
    isolated = curve.groupby("fraction")["isolated"].mean()
    frozen = estimates.loc[slug]

    fig, ax = plt.subplots(figsize=(7.2, 4.2))
    ax.fill_between(
        mean.index, low, high, color="#4c72b0", alpha=0.18, label="seed 5-95% envelope"
    )
    ax.plot(mean.index, mean, color="#4c72b0", lw=1.7, label="seed mean")
    ax.axhline(0.5, color="0.55", lw=0.9, ls=":")
    ax.axvline(
        float(frozen["critical_fraction"]),
        color="#c44e52",
        ls="--",
        lw=1.2,
        label=f"chi peak = {float(frozen['critical_fraction']):.3f}",
    )
    ax.axvline(
        float(frozen["gcc_threshold_fraction"]),
        color="#dd8452",
        ls="-.",
        lw=1.2,
        label=f"gcc 0.5 = {float(frozen['gcc_threshold_fraction']):.3f}",
    )
    current = value_at(mean, fraction)
    ax.plot([fraction], [current], "o", ms=7, color="black", zorder=6)

    info = [
        f"f = {fraction:.2f}",
        f"g(f) = {current:.3f}",
        f"seed envelope {value_at(low, fraction):.3f} to {value_at(high, fraction):.3f}",
        f"isolated stations (mean): {value_at(isolated, fraction):.1f}",
    ]
    fifth = round(fraction / 0.05) * 0.05
    on_grid = np.isclose(interval.index.to_numpy(dtype=float), fifth, atol=1e-9).any()
    if slug == "random_degree" and abs(fraction - fifth) < 1e-9 and on_grid:
        row = interval.iloc[
            int(np.argmin(np.abs(interval.index.to_numpy(dtype=float) - fifth)))
        ]
        info.append(f"interval at f = {fifth:.2f}: {row['low']:.3f} to {row['high']:.3f}")
    ax.text(
        0.02,
        0.03,
        "\n".join(info),
        transform=ax.transAxes,
        fontsize=8,
        va="bottom",
        family="monospace",
        bbox={"facecolor": "white", "edgecolor": "0.8", "alpha": 0.9, "pad": 3},
    )

    attack_label = "random" if attack == "random" else f"targeted by {measure}"
    ax.set_title(f"{component.title()} removal, {attack_label}")
    ax.set_xlabel(r"removed fraction $f$")
    ax.set_ylabel(r"largest component fraction $g(f)$")
    ax.set_xlim(0, 1)
    ax.set_ylim(-0.03, 1.05)
    ax.legend(fontsize=8, loc="upper right")
    save_copy(fig, "interactive_selection.png")
    return fig


output = widgets.interactive_output(
    collapse_view,
    {
        "component": component_toggle,
        "attack": attack_toggle,
        "measure": measure_dropdown,
        "fraction": fraction_slider,
    },
)
display(
    widgets.VBox([component_toggle, attack_toggle, measure_dropdown, fraction_slider]),
    output,
)

The widget writes the view it last drew to `figures/notebook02/interactive_selection.png`. The committed copy shows the default selection, node removal targeted by degree at $f = 0.10$.

## Critical single stations

The curves above remove many nodes at once. The vulnerability run removes one station at a time and lets the cascade spread under the frozen static capacity rule, over 81 tolerances from 0 to 2 in steps of 0.025. Every station is enumerated, which gives 86 x 81 = 6,966 trigger runs.

`containment_alpha` in `vulnerability_containment.csv` is the smallest tolerance at which the trigger stays contained, meaning only the triggered station fails (`n_failed == 1`). Lower values mark safer stations. The distribution is steep: 65 stations are first contained at $\alpha = 0.525$, Bayswater needs $\alpha = 0.55$ (the highest in the grid), and the 20 remaining stations are contained earlier. The seven terminal stations at degree one are contained at $\alpha = 0$, since a leaf carries zero betweenness load and its removal pushes no load onto anyone.

In [ ]:
containment = pd.read_csv(
    PERCOLATION_DIR / "vulnerability_containment.csv", dtype={"station_id": str}
)
tolerance = pd.read_csv(PERCOLATION_DIR / "vulnerability_tolerance_stats.csv")

print(
    f"{len(containment)} stations x {tolerance['alpha'].nunique()} tolerances"
    f" = {len(containment) * tolerance['alpha'].nunique():,} trigger runs"
)
top = containment.head(12)[
    [
        "name",
        "degree",
        "betweenness_load",
        "demand_load",
        "containment_alpha",
        "alphas_contained",
    ]
]
top

In [ ]:
top_containment = containment.head(10).iloc[::-1]
fig, axes = plt.subplots(1, 2, figsize=(11.0, 4.4))

counts = containment["containment_alpha"].value_counts().sort_index()
axes[0].bar(counts.index, counts.values, width=0.018, color="#4c72b0")
axes[0].set_title("stations by containment tolerance")
axes[0].set_xlabel(r"containment tolerance $\alpha$")
axes[0].set_ylabel("stations")

degrees = top_containment["degree"].to_numpy(dtype=float)
norm = plt.Normalize(
    vmin=float(containment["degree"].min()), vmax=float(containment["degree"].max())
)
names = [name.replace(" Stn", "") for name in top_containment["name"]]
axes[1].barh(names, top_containment["containment_alpha"], color=plt.cm.plasma(norm(degrees)))
axes[1].set_title("ten hardest stations to contain")
axes[1].set_xlabel(r"containment tolerance $\alpha$")
fig.colorbar(
    plt.cm.ScalarMappable(norm=norm, cmap="plasma"), ax=axes[1], shrink=0.85, label="degree"
)
save_copy(fig, "critical_node_ranking.png")
plt.show()

Most stations sit at the same containment tolerance, with Bayswater alone at the top. The right panel colours bars by degree, and six of the ten hardest stations have degree two. Low degree is enough to collapse the whole network. Saved copy: `figures/notebook02/critical_node_ranking.png`.

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 4.2))
for column, label in [
    ("degree_damage_rho", "degree"),
    ("betweenness_damage_rho", "betweenness"),
    ("demand_damage_rho", "demand"),
]:
    ax.plot(tolerance["alpha"], tolerance[column], lw=1.6, label=label)
ax.axvline(0.5, color="0.55", ls=":", lw=1.0, label=r"focus $\alpha = 0.5$")
ax.set_xlabel(r"tolerance $\alpha$")
ax.set_ylabel("Spearman rho with cascade damage")
ax.set_title("predictors of single-station damage")
ax.legend(fontsize=9)
save_copy(fig, "predictor_correlations.png")
plt.show()

Each line is the Spearman rank correlation between a predictor and the cascade damage over the 86 stations, computed per tolerance (`vulnerability_tolerance_stats.csv`). The problem splits into two regimes. Below the containment threshold a trigger can cascade: at $\alpha = 0$ the average trigger fails 79 of 86 stations, so most stations approach full damage, and at the focus tolerance degree leads with 0.525 against 0.376 for betweenness and 0.328 for demand. Above the threshold every trigger is contained, the remaining damage is direct, and betweenness ranks that damage at 0.9986. No single predictor leads in both regimes. Saved copy: `figures/notebook02/predictor_correlations.png`.

In [ ]:
stability = pd.read_csv(
    PERCOLATION_DIR / "vulnerability_rank_stability.csv", dtype={"station_id": str}
)
bottlenecks = pd.read_csv(
    PERCOLATION_DIR / "vulnerability_low_degree_bottlenecks.csv", dtype={"station_id": str}
)

widest = stability.sort_values("damage_rank_range", ascending=False).head(8)
print("widest rank movement across the tolerances:")
display(
    widest[
        [
            "name",
            "degree",
            "focus_damage_rank",
            "min_damage_rank",
            "max_damage_rank",
            "damage_rank_range",
            "n_top_alphas",
        ]
    ]
)
print("degree-2 stations at damage rank 1 at the focus tolerance:")
display(
    bottlenecks[
        [
            "name",
            "degree",
            "betweenness_load",
            "demand_load",
            "gcc_damage",
            "damage_rank",
            "n_failed",
        ]
    ]
)

Damage ranks move a long way with the tolerance (up to 78 places for Redcliffe), so a single-tolerance ranking needs its tolerance stated. The containment tolerance avoids that problem because it is one number per station. The degree-2 bottlenecks tie at damage rank 1 on the focus tolerance: each of them collapses all 86 stations on its own.

The committed map puts every single-station trigger on the network at the focus tolerance, coloured by final damage and sized by degree. The detail panel zooms in on the stations that do the most damage.

In [ ]:
display(Image(filename=str(FIGURES_DIR / "fig_vulnerability_map.png")))

## Summary

- Random removal halves the largest component only above 6.3% of stations. Targeted removal reaches the same point at 1.7% by degree and 1.2% by betweenness. On the edge layer the fastest attacks are betweenness and flow at 1.0%, while degree is the slowest at 6.9%.
- The susceptibility peak and the 0.5 crossing disagree on this network, so a collapse claim has to say which estimate it quotes.
- The single-station response is steep: 66 of 86 stations are first contained at a tolerance of 0.525 or higher, and Bayswater needs 0.55. Degree predicts cascade damage best at the focus tolerance (0.525), betweenness ranks the direct damage above the threshold at 0.9986, and degree two is enough for a station to collapse the whole network.
- This is a tree, so the sharp response follows from the topology. Notebook 03 covers the cascade side with avalanche statistics, and notebook 04 covers recovery and bus strategies.

Regenerate the curves and tables with `make percolation PYTHON=.venv/bin/python`; the notebook reads the result files and copies its figures to `figures/notebook02/`.